# SoccerNet Dense Video Captioning: chạy trên Colab (v1.8)

Xem `CHANGELOG.md` để biết thay đổi của từng phiên bản.

Notebook này train và test dự án **soccernet-caption-deltalab** (QFormer + GPT-2) trên một phần nhỏ của dataset SoccerNet `caption-2024`.

**Chuẩn bị trước khi chạy:**
1. Chọn GPU: `Runtime → Change runtime type → A100 GPU` hoặc `T4 GPU`.
2. Chạy lần lượt từng cell từ trên xuống. Cell 4 tự `git clone` code từ GitHub.

**Khi code trên GitHub có bản mới:** chỉ cần chạy lại **cell 4** (`git pull`), rồi chạy lại cell 8. Dữ liệu đã tải không bị mất.

Dữ liệu (feature và nhãn) là công khai: notebook tải thẳng từ HuggingFace về ổ của Colab, không cần NDA hay token. Model đã train được lưu vào Drive, nên vẫn còn sau khi hết phiên Colab.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cấu hình
Sửa các giá trị ở đây. Đặt `-1` để lấy toàn bộ split: train 281, valid 92, test 98 trận, khoảng 89 GB.

In [ ]:
GITHUB_REPO = "https://github.com/ThuLinh3009/soccer_project.git"  # repo code
GIT_BRANCH = "main"
DRIVE_MODEL_DIR = "/content/drive/MyDrive/soccer_project/sn-models"            # nơi lưu model đã train

N_TRAIN, N_VALID, N_TEST = 20, 5, 5  # số trận mỗi split (~190 MB/trận), 30 trận ~5,7 GB

MODEL_NAME = "gpt2-30games" # tên thư mục kết quả trong models/
GPT_TYPE = "gpt2"           # "gpt2" (124M) hoặc "gpt2-medium" (355M)
MAX_EPOCHS = 10             # số epoch train caption; README dùng mặc định 15
EVAL_FREQ = 5               # chấm METEOR/BLEU trên valid mỗi N epoch (sinh câu chậm, đừng để 1)
BATCH_SIZE = 32
NUM_WORKERS = 2             # Colab chỉ có ít CPU, để 4 sẽ bị cảnh báo chậm
NMS_THRESHOLD = 0.7

## 3. Mount Google Drive

Khi cửa sổ xin quyền hiện ra, chọn đúng tài khoản Google rồi bấm **Allow / Cho phép** cho mọi quyền được hỏi.
Nếu vẫn báo `mount failed`, bạn có thể bỏ qua: model sẽ được lưu tạm trên Colab, và cell 9 có cách tải về máy.

In [ ]:
from google.colab import drive
try:
    drive.mount("/content/drive", force_remount=True)
    USE_DRIVE = True
except Exception as e:
    USE_DRIVE = False
    print("Không mount được Drive:", e)
    print("Tiếp tục chạy, model sẽ lưu tạm trong /content (mất khi hết phiên Colab).")

## 4. Lấy code từ GitHub
Lần đầu thì `git clone`. Các lần sau thì `git pull` để cập nhật code mới nhất, còn `dataset/` và `models/` vẫn giữ nguyên.

In [ ]:
import os, glob
WORK_DIR = "/content/soccernet-caption-deltalab"
if os.path.isdir(f"{WORK_DIR}/.git"):
    !git -C {WORK_DIR} fetch -q origin && git -C {WORK_DIR} reset -q --hard origin/{GIT_BRANCH}
else:
    !git clone -q -b {GIT_BRANCH} {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -3
!grep -m1 "^## v" CHANGELOG.md

## 5. Cài thư viện

- `nlg-eval` được cài với `--no-deps` vì các thư viện phụ thuộc cũ của nó (gensim 3.8, Theano) không cài được trên Python mới. Code chỉ dùng BLEU, METEOR, ROUGE và CIDEr nên không cần đến chúng.
- METEOR cần Java, nên cell này cài Java nếu Colab chưa có.
- Java trên Colab in thêm các dòng cảnh báo `Cgroup ...` vào stdout, làm METEOR đọc nhầm điểm (`ValueError: could not convert string to float`). `JAVA_TOOL_OPTIONS` tắt các cảnh báo này.

In [ ]:
!pip install -q SoccerNet tiktoken transformers wandb pycocoevalcap psutil six
!pip install -q --no-deps git+https://github.com/Maluuba/nlg-eval.git@master
!which java || (apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null)

import os
os.environ["JAVA_TOOL_OPTIONS"] = "-XX:-UseContainerSupport -Xlog:disable"
!java -version

Kiểm tra nhanh METEOR. Nếu cell này in ra một con số thì bước chấm điểm sẽ chạy được.

In [ ]:
from nlgeval.pycocoevalcap.meteor.meteor import Meteor
m = Meteor()
print("METEOR test:", m.compute_score({0: ["a player takes a free kick"]}, {0: ["a player is awarded a free kick"]})[0])
del m

## 6. Tải dữ liệu (feature + nhãn) từ HuggingFace

In [ ]:
!python download_subset.py dataset/caption-2024 --train {N_TRAIN} --valid {N_VALID} --test {N_TEST}
!du -sh dataset

## 7. Lưu `models/` thẳng lên Drive
Tạo symlink `models → Drive`, để checkpoint và kết quả không bị mất khi hết phiên Colab. Nếu chưa mount được Drive thì `models/` nằm trên ổ Colab.

In [ ]:
if USE_DRIVE:
    os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
    if not os.path.islink("models"):
        !rm -rf models
    !ln -sfn "{DRIVE_MODEL_DIR}" models
else:
    os.makedirs("models", exist_ok=True)
!ls -la models

## 8. Train và test

`main.py` lần lượt chạy:
1. **classifying**: 10 epoch, phân loại sự kiện.
2. **captioning**: `MAX_EPOCHS` epoch, sinh lời bình bằng GPT-2.
3. **spotting**: 15 epoch, tìm thời điểm xảy ra sự kiện.
4. **dvc**: ghép bước 2 và 3, rồi chấm METEOR/BLEU/CIDEr trên test.

Hai biến môi trường được bật:
- `SN_LOCAL_SUBSET=1`: chỉ dùng các trận đã tải.
- `WANDB_MODE=offline`: không cần tài khoản wandb.

In [ ]:
os.environ["SN_LOCAL_SUBSET"] = "1"
os.environ["WANDB_MODE"] = "offline"
os.environ["JAVA_TOOL_OPTIONS"] = "-XX:-UseContainerSupport -Xlog:disable"  # để METEOR đọc đúng điểm

!python main.py     --SoccerNet_path dataset/caption-2024     --model_name {MODEL_NAME}     --GPU 0     --pool QFormer     --model_type gpt     --gpt_type {GPT_TYPE}     --NMS_threshold {NMS_THRESHOLD}     --split_test test     --max_epochs {MAX_EPOCHS}     --evaluation_frequency {EVAL_FREQ}     --batch_size {BATCH_SIZE} \
    --max_num_worker {NUM_WORKERS}

## 9. Xem kết quả

In [ ]:
import glob
log = sorted(glob.glob(f"models/{MODEL_NAME}/*.log"))[-1]
print("Log:", log)
!grep -E "METEOR|Bleu_4|CIDEr|a_mAP" "{log}" | tail -30

In [ ]:
# Xem vài câu bình luận model sinh ra cho trận test đầu tiên
import json
files = sorted(glob.glob(f"models/{MODEL_NAME}/outputs/test/**/results_dense_captioning.json", recursive=True))      or sorted(glob.glob(f"models/{MODEL_NAME}/outputs/**/*.json", recursive=True))
print(files[:5])
if files:
    preds = json.load(open(files[0]))
    for p in preds.get("predictions", preds.get("annotations", []))[:15]:
        print(p.get("gameTime"), "|", p.get("label"), "|", p.get("comment", p.get("anonymized")))

Nếu không mount được Drive: nén kết quả (log, output, checkpoint caption) và tải về máy trước khi hết phiên Colab.

In [ ]:
if not USE_DRIVE:
    !cd models && zip -qr /content/{MODEL_NAME}_results.zip {MODEL_NAME} -x "*/classifying/*" "*/spotting/*"
    from google.colab import files as colab_files
    colab_files.download(f"/content/{MODEL_NAME}_results.zip")
else:
    print("Kết quả đã nằm trên Drive:", DRIVE_MODEL_DIR)